In [1]:
import os
import json
import pickle

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

from xgboost import XGBClassifier

In [2]:
MASTER = "/kaggle/input/notebooks/shri7ul/13-semantic-features-ipynb/master_semantic.parquet"

MODEL_JSON = "/kaggle/input/notebooks/shri7ul/16-final-xgboost-ipynb/xgb_model.json"

FEATURE_COLUMNS = "/kaggle/input/notebooks/shri7ul/16-final-xgboost-ipynb/feature_columns.pkl"

CATEGORY_MAPPING = "/kaggle/input/notebooks/shri7ul/16-final-xgboost-ipynb/category_mapping.pkl"

PCA_PATH = "/kaggle/input/notebooks/shri7ul/13-semantic-features-ipynb/objective_pca.pkl"

SEMANTIC_CONFIG = "/kaggle/input/notebooks/shri7ul/13-semantic-features-ipynb/semantic_config.json"

In [3]:
# ==========================================================
# Load Assets
# ==========================================================

master = pd.read_parquet(MASTER)

print("Master Shape :", master.shape)

with open(FEATURE_COLUMNS, "rb") as f:
    feature_columns = pickle.load(f)

print("Features :", len(feature_columns))

with open(CATEGORY_MAPPING, "rb") as f:
    category_mapping = pickle.load(f)

print("Category Mapping Loaded")

with open(PCA_PATH, "rb") as f:
    pca = pickle.load(f)

print("PCA Loaded")

with open(SEMANTIC_CONFIG, "r") as f:
    semantic_config = json.load(f)

print(semantic_config)

model = XGBClassifier()

model.load_model(MODEL_JSON)

print("XGBoost Model Loaded")

Master Shape : (35072, 154)
Features : 145
Category Mapping Loaded
PCA Loaded
{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dimension': 384, 'pca_dimension': 32}
XGBoost Model Loaded


In [4]:
# ==========================================================
# Load Embedding Model
# ==========================================================

HF_MODEL = "/kaggle/input/huggingface-models/sentence-transformers/all-MiniLM-L6-v2"

try:

    embedder = SentenceTransformer(HF_MODEL)

except:

    embedder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [5]:
# ==========================================================
# Utility Functions
# ==========================================================

import re
import math
import string

from collections import Counter


def safe_div(a, b):

    if b == 0:
        return 0.0

    return a / b


def count_digits(text):

    return sum(ch.isdigit() for ch in text)


def count_upper(text):

    return sum(ch.isupper() for ch in text)


def count_punctuation(text):

    return sum(ch in string.punctuation for ch in text)


def word_count(text):

    return len(text.split())


def char_count(text):

    return len(text)


def avg_word_length(text):

    words = text.split()

    if len(words) == 0:
        return 0.0

    return np.mean([len(w) for w in words])


def log1p(x):

    return math.log1p(max(x, 0))


def normalize_text(text):

    if pd.isna(text):
        return ""

    return str(text)


def contains_number(text):

    return int(bool(re.search(r"\d", text)))


def contains_percent(text):

    return int("%" in text)


def contains_decimal(text):

    return int(bool(re.search(r"\d+\.\d+", text)))


def contains_fraction(text):

    return int(bool(re.search(r"\d+\s*/\s*\d+", text)))


def contains_money(text):

    patterns = [

        "$",
        "৳",
        "tk",
        "taka",
        "usd",
        "dollar"

    ]

    text = text.lower()

    return int(any(p in text for p in patterns))


def contains_multiply(text):

    return int(bool(re.search(r"\bx\b|\*", text.lower())))


def contains_division(text):

    return int("/" in text)


def contains_add(text):

    return int("+" in text)


def contains_subtract(text):

    return int("-" in text)


def contains_angle(text):

    return int("angle" in text.lower())


def contains_shape(text):

    keywords = [

        "triangle",
        "square",
        "rectangle",
        "circle",
        "polygon",
        "shape"

    ]

    text = text.lower()

    return int(any(k in text for k in keywords))


def contains_table(text):

    return int("table" in text.lower())


def contains_area(text):

    return int("area" in text.lower())


def contains_volume(text):

    return int("volume" in text.lower())


def contains_length(text):

    return int("length" in text.lower())


def contains_probability(text):

    return int("probability" in text.lower())